In [1]:
import pandas as pd
import numpy as np

print("Environment working")

Environment working


In [5]:
import pandas as pd
import numpy as np

# File paths
review_path = "../data/raw/yelp_academic_dataset_review.json"
business_path = "../data/raw/yelp_academic_dataset_business.json"
user_path = "../data/raw/yelp_academic_dataset_user.json"
print("Paths loaded successfully")

Paths loaded successfully


In [ ]:
# STEP 3
# Loading dataframes
# For initial exploration, we will load a sample of the reviews and users datasets to avoid memory issues, while loading the entire business dataset to analyze restaurant-related information.
# This approach allows us to get a sense of the structure and content of the datasets without overwhelming our system's memory, while still providing us with enough data to perform meaningful analysis on restaurant reviews and business information.

reviews_sample = pd.read_json(
    review_path,
    lines=True,
    nrows=5000
)

business_df = pd.read_json(
    business_path,
    lines=True,
)

users_sample = pd.read_json(
    user_path,
    lines=True,
    nrows=5000
)

print("Reviews shape:", reviews_sample.shape)
print("Business dataset shape:")
print(business_df.shape)
print("Users shape:", users_sample.shape)

Reviews shape: (5000, 9)
Business dataset shape:
(150346, 14)
Users shape: (5000, 22)


In [ ]:
# STEP 4
# Inspecting columns of each dataset
# This will help us understand the structure of the data and identify key features for analysis.

print("REVIEWS COLUMNS")
print(reviews_sample.columns)

print("\nBUSINESS COLUMNS")
print(business_df.columns)

print("\nUSER COLUMNS")
print(users_sample.columns)

REVIEWS COLUMNS
Index(['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny',
       'cool', 'text', 'date'],
      dtype='str')

BUSINESS COLUMNS
Index(['business_id', 'name', 'address', 'city', 'state', 'postal_code',
       'latitude', 'longitude', 'stars', 'review_count', 'is_open',
       'attributes', 'categories', 'hours'],
      dtype='str')

USER COLUMNS
Index(['user_id', 'name', 'review_count', 'yelping_since', 'useful', 'funny',
       'cool', 'elite', 'friends', 'fans', 'average_stars', 'compliment_hot',
       'compliment_more', 'compliment_profile', 'compliment_cute',
       'compliment_list', 'compliment_note', 'compliment_plain',
       'compliment_cool', 'compliment_funny', 'compliment_writer',
       'compliment_photos'],
      dtype='str')


In [ ]:
# STEP 5
# Displaying first few rows of each dataset to get a sense of the data
reviews_sample.head(2)

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18


In Reviews, there are important fields like: user_id, business_id, stars, text, date

These serves as our behavioral signal layer.

In [ ]:
# STEP 6
# Displaying first few rows of business dataset

business_df.head(2)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."


In Businesses

Fields like: categories,city, attributes,stars

This serves as our contextual environment layer.

In [ ]:

# STEP 7
# Displaying first few rows of users dataset

users_sample.head(2)

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946


In Users fields like: review_count, average_stars, fans

These serves as our behavioral metadata layer.

STEP 8 
Filtering Restaurant Businesses

We only want businesses related to: restaurants, food, cafes, bars because this domain contains rich emotional/social behavior.

In [21]:
# Keep only businesses with restaurant-related categories
# This will help us focus our analysis on the restaurant industry, which is a major part of Yelp's business and user interactions.
# filters businesses whose categories contain: Restaurant, Food, Coffee, Cafe, Bar

restaurant_businesses = business_df[
    business_df["categories"]
    .fillna("")
    .str.contains(
        "Restaurant|Food|Coffee|Cafe|Bar",
        case=False,
        regex=True
    )
]

print("Restaurant businesses:")
print(restaurant_businesses.shape)

Restaurant businesses:
(68696, 14)


In [ ]:
# STEP 9
# Inspect Restaurant businesses

restaurant_businesses[
    ["business_id", "name", "categories", "city"]
].head(10)

,business_id,name,categories,city
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",Philadelphia
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,"Brewpubs, Breweries, Food",Green Lane
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",Ashland City
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,"Pubs, Restaurants, Italian, Bars, American (Tr...",Affton
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",Nashville
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,"Vietnamese, Food, Restaurants, Food Trucks",Tampa Bay
12,il_Ro8jwPlHresjw9EGmBg,Denny's,"American (Traditional), Restaurants, Diners, B...",Indianapolis
14,0bPLkL0QhhPO5kt1_EXmNQ,Zio's Italian Market,"Food, Delis, Italian, Bakeries, Restaurants",Largo
15,MUTTqe8uqyMdBl186RmNeA,Tuna Bar,"Sushi Bars, Restaurants, Japanese",Philadelphia
19,ROeacJQwBeh05Rqg7F6TCg,BAP,"Korean, Restaurants",Philadelphia


In [ ]:
# STEP 10
# Extract unique restaurant business IDs

restaurant_ids = set(
    restaurant_businesses["business_id"]
)

print("Number of restaurant IDs:")
print(len(restaurant_ids))

Number of restaurant IDs:
68696


In [ ]:
# STEP 11
# Filter reviews to include only those related to the restaurant businesses identified above. 
# This will allow us to analyze user feedback specifically for restaurants, which is crucial for understanding customer satisfaction and business performance in this sector.
# This makes personas cleaner, preferences clearer, recommendations better

restaurant_reviews = reviews_sample[
    reviews_sample["business_id"].isin(
        restaurant_businesses["business_id"]
    )
]

print("Restaurant reviews shape:")
print(restaurant_reviews.shape)

Restaurant reviews shape:
(1359, 9)


In [ ]:
# STEP 12
#  Inspect restaurant reviews
# This will give us insights into the types of feedback customers are providing for restaurants, which can inform our analysis of customer satisfaction and business performance in this sector.

restaurant_reviews[
    ["user_id", "business_id", "stars", "text"]
].head(5)

,user_id,business_id,stars,text
0,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,"If you decide to eat here, just be aware it is..."
5,eUta8W_HdHMXPzLBBZhL1A,04UD14gamNjLY0IDYVhHJg,1,I am a long term frequent customer of this est...
7,yfFzsLmaWF2d4Sr0UNbBgg,LHSTtnW3YHCeUkRDGyJOyw,5,Amazingly amazing wings and homemade bleu chee...
11,ZbqSHbgCjzVAqaa7NKWn5A,EQ-TZ2eeD_E0BHuvoaeG5Q,4,"Locals recommended Milktooth, and it's an amaz..."
12,9OAtfnWag-ajVxRbUTGIyg,lj-E32x9_FA7GmUrBGBEWg,4,Love going here for happy hour or dinner! Gre...


Finally, I am looking at real behavioral data. I can now see:

emotions
complaints
enthusiasm
sarcasm
value judgments

This is a raw psychological signal.

STEP 13: Find Sweet-Spot Users

Now we identify behaviorally rich users.

In [18]:
# Count reviews per user
# This will help us identify active users and understand the distribution of reviews among users, which can inform our analysis of user behavior and preferences in the restaurant sector.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

# Keep users with 15–50 reviews

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

Sweet-spot users:
(0, 2)


In [ ]:
# STEP 13B
# Inspect sweet-spot users
sweet_spot_users.head(10)

,user_id,review_count


In [ ]:
# STEP 14
# Read review data in chunks
# This approach allows us to process large datasets without running into memory issues, enabling us to filter and analyze reviews related to restaurants efficiently.
# By reading the review data in chunks, we can handle the large size of the dataset while still extracting relevant information for our analysis of restaurant reviews.

chunk_size = 100000

restaurant_review_chunks = []

for chunk in pd.read_json(
    review_path,
    lines=True,
    chunksize=chunk_size
):
    
    filtered_chunk = chunk[
        chunk["business_id"].isin(restaurant_ids)
    ]
    
    restaurant_review_chunks.append(filtered_chunk)

    print(
        f"Processed chunk with "
        f"{len(filtered_chunk)} restaurant reviews"
    )

Processed chunk with 80537 restaurant reviews
Processed chunk with 80648 restaurant reviews
Processed chunk with 79453 restaurant reviews
Processed chunk with 75590 restaurant reviews
Processed chunk with 71692 restaurant reviews
Processed chunk with 70100 restaurant reviews
Processed chunk with 68485 restaurant reviews
Processed chunk with 79838 restaurant reviews
Processed chunk with 81158 restaurant reviews
Processed chunk with 80883 restaurant reviews
Processed chunk with 77279 restaurant reviews
Processed chunk with 72916 restaurant reviews
Processed chunk with 70352 restaurant reviews
Processed chunk with 68087 restaurant reviews
Processed chunk with 78749 restaurant reviews
Processed chunk with 80778 restaurant reviews
Processed chunk with 80846 restaurant reviews
Processed chunk with 77760 restaurant reviews
Processed chunk with 73650 restaurant reviews
Processed chunk with 69105 restaurant reviews
Processed chunk with 69018 restaurant reviews
Processed chunk with 77708 restaur

In [ ]:
# Step 15
# Concatenate all filtered chunks into a single DataFrame
# This will give us a complete dataset of restaurant reviews that we can use for further analysis, 
# such as sentiment analysis, user behavior analysis, and business performance evaluation in the restaurant sector.
# This is our real behavioral corpus.

restaurant_reviews = pd.concat(
    restaurant_review_chunks,
    ignore_index=True
)

print("Final restaurant reviews shape:")
print(restaurant_reviews.shape)

Final restaurant reviews shape:
(5259993, 9)


In [ ]:
# STEP 16
# Count reviews per user
# This will help us identify users who have reviewed a moderate number of restaurants, which might be indicative of engaged reviewers.

user_review_counts = (
    restaurant_reviews
    .groupby("user_id")
    .size()
    .reset_index(name="review_count")
)

sweet_spot_users = user_review_counts[
    (user_review_counts["review_count"] >= 15) &
    (user_review_counts["review_count"] <= 50)
]

print("Sweet-spot users:")
print(sweet_spot_users.shape)

Sweet-spot users:
(40884, 2)


In [ ]:
# STEP 17
# Curate a sample of sweet-spot users for further analysis, 
# ensuring we have a manageable number of users to work with while still capturing a representative subset of engaged reviewers.

curated_users = sweet_spot_users.sample(
    n=300,
    random_state=42
)

print(curated_users.shape)

(300, 2)


In [ ]:
# Step 18
# Filter restaurant reviews to include only those from the curated sweet-spot users.
# This will allow us to focus our analysis on a specific subset of engaged users, which can provide more meaningful insights into user behavior and preferences in the restaurant sector.

curated_user_ids = set(
    curated_users["user_id"]
)

curated_reviews = restaurant_reviews[
    restaurant_reviews["user_id"]
    .isin(curated_user_ids)
]

print("Curated reviews shape:")
print(curated_reviews.shape)

Curated reviews shape:
(7413, 9)


This reveals a curated corpus of: psychologically rich users, restaurant behaviors, emotional language, preferences, review styles

Enough to build: personas, review simulation, recommendations, conversational reasoning

In [28]:
# STEP 19
# Transforming raw reviews into interpretable human behavioral traits.
# First, we organize and group reviews by user and business, then we can analyze patterns in the review text, star ratings, and other features to derive insights about user preferences, sentiment, and engagement with restaurants. This will help us understand the underlying behaviors and traits of users in the context of restaurant reviews.



user_histories = (
    curated_reviews
    .groupby("user_id")
    .agg({
        "text": list,
        "stars": list,
        "business_id": list,
        "date": list
    })
    .reset_index()
)

print("User histories shape:")
print(user_histories.shape)

User histories shape:
(300, 5)


In [ ]:
# STEP 20
# Inspecting a sample user history to understand the structure of the data and the type of information we have for each user, which will help us in deriving behavioral traits and insights from their reviews.
# Change the index to inspect different users and their review histories.

sample_user = user_histories.iloc[70]

print("USER ID:")
print(sample_user["user_id"])

print("\nSTAR RATINGS:")
print(sample_user["stars"][:5])

print("\nFIRST REVIEW:")
print(sample_user["text"][0][:500])

USER ID:
Gcxm0XlnMIW0sUwiYgo4dA

STAR RATINGS:
[4, 5, 5, 4, 5]

FIRST REVIEW:
Went here for Valentine's day. Very crowded but our reservation was on time and server was good. Grilled seafood app was the highlight of the meal. Grilled scallops, calamari, shrimp and cherry tomatoes over arugula perfectly dressed. 5 stars.
Had filet and pescatore for entree. Filet was good but not amazing- 3 stars. Pescatore was delicious- 5 stars. Lobster slightly dry, but  calamari and scallops and garlic white wine sauce were more than enough.
Dessert was a hit and a miss. Chocolate souff


By changing the index, i notice behavioral traces of a real human.

Things i notice:

tone
complaints
enthusiasm
emotional intensity
writing style
priorities

This is where personas emerge.

In [ ]:
# STEP 21 — Creating First Persona Features
# Here we are calculating basic features for each user based on their reviews, such as average rating, rating variance, review count, and average review length. 
# These features can help us understand user behavior and preferences in the context of restaurant reviews, which can be useful for building user personas and improving recommendation systems.

persona_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_rating=("stars", "mean"),
        rating_variance=("stars", "std"),
        review_count=("stars", "count"),
        avg_review_length=(
            "text",
            lambda x: np.mean(
                x.str.len()
            )
        )
    )
    .reset_index()
)

print(persona_features.shape)

persona_features.head()

(300, 5)


,user_id,avg_rating,rating_variance,review_count,avg_review_length
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455


WHAT THESE FEATURES MEAN
  Feature	               Psychological Meaning
- avg_rating	           harsh vs lenient
- rating_variance	       emotional consistency
- review_count	           engagement
- avg_review_length	       verbosity/detail orientation

These are behavioral signals.

In [ ]:
# STEP 22 — Adding Sentiment Features
# Now we estimate emotional tone.

In [33]:
from textblob import TextBlob
from tqdm import tqdm

tqdm.pandas()

In [34]:
# Sentiment per review

curated_reviews["sentiment"] = (
    curated_reviews["text"]
    .progress_apply(
        lambda x: TextBlob(x).sentiment.polarity
    )
)

100%|██████████| 7413/7413 [00:07<00:00, 975.54it/s] 


In [ ]:
# STEP 23
# Aggregating User Sentiment

sentiment_features = (
    curated_reviews
    .groupby("user_id")
    .agg(
        avg_sentiment=("sentiment", "mean"),
        sentiment_variance=("sentiment", "std")
    )
    .reset_index()
)

sentiment_features.head()

,user_id,avg_sentiment,sentiment_variance
0,-EX1hrPRBqNkVavtMllTCA,0.138507,0.222461
1,-M7fUg7FrdGctKr5f_eMUQ,0.215294,0.162739
2,-WM58wLjtlHlR91xVfM1FQ,0.298114,0.206382
3,-qTtg1D3RidRa4cTB-ftwg,0.425713,0.245910
4,02H49g16MdRoZKoX6IEoFA,0.255776,0.355440


WHAT THESE MEAN
Feature	              Meaning
avg_sentiment	      positivity/negativity
sentiment_variance	  emotional stability

In [36]:
# STEP 24
# Merge Persona Features

persona_df = persona_features.merge(
    sentiment_features,
    on="user_id"
)

print(persona_df.shape)

persona_df.head()

(300, 7)


,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440


In [ ]:
# STEP 25 — Interpret Personas

persona_df.describe()

,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
count,300.000000,300.000000,300.000000,300.000000,300.000000,300.000000
mean,3.884222,1.081557,24.710000,592.957436,0.256042,0.186791
std,0.604832,0.363819,9.462082,367.794262,0.087883,0.067733
min,1.591837,0.000000,15.000000,175.900000,-0.071841,0.066279
25%,3.523810,0.834853,17.000000,346.679412,0.198519,0.138404
50%,3.942810,1.095445,22.000000,499.306548,0.255502,0.172593
75%,4.315789,1.335010,31.000000,751.808527,0.314091,0.225523
max,5.000000,1.999557,50.000000,2534.300000,0.510412,0.430499


This uncovers human archetypes.

Examples:

angry critics
generous reviewers
emotional storytellers
concise pragmatists

In [ ]:
# Harsh Users
# These users tend to give lower ratings on average, which may indicate a more critical perspective or higher standards when it comes to restaurant experiences.

persona_df.sort_values(
    by="avg_rating"
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
20,3GKk2POn0VFznH_KuRX8UA,1.591837,1.153227,49,452.938776,0.034711,0.254318
7,0Tsu6-uhw_w9Z3W0Xlnazg,1.687500,1.138347,16,411.125000,0.093389,0.225517
199,gVvkw-bW7hcrs9P5xfwOhw,1.708333,1.366658,24,502.041667,0.031524,0.212126
179,cb5omh0nibWUYN2rws5rdg,1.850000,1.182103,20,529.450000,-0.071841,0.202567
225,mD-IgInk0o8pXrJI-P8pYA,2.266667,1.099784,15,297.333333,0.124339,0.143346


In [ ]:
# Lenient Users
# These users tend to give higher ratings on average, which may indicate a more positive outlook or a tendency to be more forgiving in their reviews.

persona_df.sort_values(
    by="avg_rating",
    ascending=False
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
212,jYOK8bu9lIxkVoKvwPC7ig,5.000000,0.000000,31,361.774194,0.382680,0.176973
81,I6x-ZBHeCNlMnmtfUVf5lg,5.000000,0.000000,24,176.208333,0.510412,0.158668
167,_m0Sxb2_kUFtic4wGwIGzw,4.960000,0.200000,25,249.040000,0.485915,0.160752
5,0650daOKAuufqymyOBe3cA,4.937500,0.250000,16,255.937500,0.478739,0.196268
23,4Z2lfaP3d3oOKmEfmc9PCw,4.933333,0.258199,15,828.200000,0.315061,0.154081


In [ ]:
# Verbose Users
# These users tend to write longer reviews, which may indicate a higher level of engagement or a desire to provide more detailed feedback.

persona_df.sort_values(
    by="avg_review_length",
    ascending=False
).head(5)

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance
168,_p7aWe_YiAZW2m6RnmAMVA,3.325000,1.071484,40,2534.300000,0.121472,0.099103
247,q4Oz1c_OjGu_a8ZkP53Vnw,2.723404,1.378438,47,2529.531915,0.118203,0.093142
54,Do8oscF3LjCl-_pXrmgLXA,4.041667,0.954585,24,2228.416667,0.178727,0.066896
19,2xNYCJxZOrhVsrVMz8mlDQ,3.911765,0.933149,34,2181.441176,0.228747,0.066279
38,9VdyNBdQbaZTrFDmD49e7A,3.736842,1.097578,19,2119.368421,0.181577,0.117300


With these, we can:

characterize users
compare personalities
simulate tendencies
reason about preferences

Next, We will transform numeric persona signals into recognizable human archetypes.

Examples:

harsh critic
emotional foodie
soft-life explorer
concise pragmatist
luxury seeker

This becomes our Dynamic Cognitive Persona Layer.

In [ ]:
# STEP 26 — Prepare for Clustering
# Here we are selecting the relevant features for clustering and filling any missing values with 0. 
# This will allow us to group users into distinct personas based on their review behavior and sentiment, which can be useful for targeted marketing, personalized recommendations, and understanding customer segments in the restaurant industry.

clustering_features = persona_df[
    [
        "avg_rating",
        "rating_variance",
        "avg_review_length",
        "avg_sentiment",
        "sentiment_variance"
    ]
].fillna(0)

clustering_features.head()

,avg_rating,rating_variance,avg_review_length,avg_sentiment,sentiment_variance
0,3.250000,1.441725,390.194444,0.138507,0.222461
1,4.090909,1.341963,358.727273,0.215294,0.162739
2,4.041667,1.197068,632.666667,0.298114,0.206382
3,4.733333,0.703732,195.666667,0.425713,0.245910
4,4.272727,1.202451,406.545455,0.255776,0.355440


In [ ]:
# STEP 27 — Standardize Features / Scale Features for Clustering
# Standardizing features is crucial for clustering algorithms, especially those that rely on distance metrics (like K-Means), as it ensures that all features contribute equally to the distance calculations.

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    clustering_features
)

print(scaled_features.shape)

(300, 5)


In [43]:
# STEP 28 — Create Behavioral Clusters with K-Means
# K-Means is a popular clustering algorithm that partitions data into K distinct clusters based on feature similarity. 
# By applying K-Means to our standardized features, we can identify distinct user personas based on their review behavior and sentiment, which can provide valuable insights for targeted marketing and personalized recommendations in the restaurant industry.

from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=5,
    random_state=42
)

persona_df["cluster"] = kmeans.fit_predict(
    scaled_features
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1


WHY THIS MATTERS

The model is now grouping users by:

emotional style
harshness
verbosity
behavioral consistency

This is latent human behavior discovery.

In [ ]:
# STEP 29 — Analyze Cluster Distribution. 
# This will help us understand how users are grouped into different personas based on their review behavior and sentiment, 
# which can inform our strategies for targeted marketing, personalized recommendations, and customer segmentation in the restaurant industry.

persona_df["cluster"].value_counts()

cluster
3    101
0     83
1     54
2     46
4     16
Name: count, dtype: int64

In [ ]:
# STEP 30 — Summarize Cluster Characteristics
# This will allow us to understand the defining features of each cluster, which can help us interpret the underlying behaviors 
# and traits of users in each persona, providing insights that can inform targeted marketing strategies, personalized recommendations, and customer segmentation in the restaurant industry.
# Understand Each Cluster AND compute average traits per cluster.

cluster_summary = (
    persona_df
    .groupby("cluster")
    [
        [
            "avg_rating", # 
            "avg_review_length", 
            "avg_sentiment",
            "rating_variance"
        ]
    ]
    .mean()
)

cluster_summary

,avg_rating,avg_review_length,avg_sentiment,rating_variance
cluster,,,,
0,4.464400,411.778871,0.340184,0.730411
1,3.749031,345.437584,0.283938,1.425644
2,2.947635,572.800835,0.139540,1.428969
3,3.908610,708.455119,0.235566,1.055194
4,3.869562,1697.071850,0.189596,0.909445


In [46]:
# STEP 31 - Assign Human Archetype Names
# Interpret Clusters with Names


cluster_names = {
    0: "Warm Optimist",
    1: "Reactive Reviewer",
    2: "Harsh Critic",
    3: "Emotional Storyteller",
    4: "Deep Experience Analyst"
}

persona_df["archetype"] = (
    persona_df["cluster"]
    .map(cluster_names)
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer


In [47]:
# STEP 31A - Create Structured Behavioral Descriptors
# 

behavioral_descriptors = {

    0: {
        "archetype": "Warm Optimist",

        "traits": {
            "positivity": "high",
            "verbosity": "moderate",
            "emotional_stability": "stable",
            "review_style": "supportive",
            "expectation_level": "moderate",
            "decision_style": "emotionally positive"
        }
    },

    1: {
        "archetype": "Reactive Reviewer",

        "traits": {
            "positivity": "mixed",
            "verbosity": "moderate",
            "emotional_stability": "volatile",
            "review_style": "emotion-driven",
            "expectation_level": "variable",
            "decision_style": "experience-sensitive"
        }
    },

    2: {
        "archetype": "Harsh Critic",

        "traits": {
            "positivity": "low",
            "verbosity": "high",
            "emotional_stability": "critical",
            "review_style": "analytical",
            "expectation_level": "high",
            "decision_style": "detail-oriented"
        }
    },

    3: {
        "archetype": "Emotional Storyteller",

        "traits": {
            "positivity": "moderate",
            "verbosity": "high",
            "emotional_stability": "reflective",
            "review_style": "narrative",
            "expectation_level": "balanced",
            "decision_style": "emotionally expressive"
        }
    },

    4: {
        "archetype": "Deep Experience Analyst",

        "traits": {
            "positivity": "moderate",
            "verbosity": "very high",
            "emotional_stability": "stable",
            "review_style": "deeply descriptive",
            "expectation_level": "high",
            "decision_style": "deliberative"
        }
    }
}

In [48]:
# STEP 31B — Attach Archetype Names
persona_df["archetype"] = (
    persona_df["cluster"]
    .apply(
        lambda x: behavioral_descriptors[x]["archetype"]
    )
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer


In [49]:
# STEP 31C — Attach Structured Traits
# Add the full trait dictionaries.


persona_df["behavior_profile"] = (
    persona_df["cluster"]
    .apply(
        lambda x: behavioral_descriptors[x]["traits"]
    )
)

persona_df.head()

,user_id,avg_rating,rating_variance,review_count,avg_review_length,avg_sentiment,sentiment_variance,cluster,archetype,behavior_profile
0,-EX1hrPRBqNkVavtMllTCA,3.250000,1.441725,36,390.194444,0.138507,0.222461,2,Harsh Critic,"{'positivity': 'low', 'verbosity': 'high', 'em..."
1,-M7fUg7FrdGctKr5f_eMUQ,4.090909,1.341963,22,358.727273,0.215294,0.162739,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'..."
2,-WM58wLjtlHlR91xVfM1FQ,4.041667,1.197068,24,632.666667,0.298114,0.206382,3,Emotional Storyteller,"{'positivity': 'moderate', 'verbosity': 'high'..."
3,-qTtg1D3RidRa4cTB-ftwg,4.733333,0.703732,15,195.666667,0.425713,0.245910,0,Warm Optimist,"{'positivity': 'high', 'verbosity': 'moderate'..."
4,02H49g16MdRoZKoX6IEoFA,4.272727,1.202451,22,406.545455,0.255776,0.355440,1,Reactive Reviewer,"{'positivity': 'mixed', 'verbosity': 'moderate..."


WHAT THE DATAFRAME NOW CONTAINS

Each user now has:

Column	                Meaning
avg_rating	            rating behavior
avg_review_length	    verbosity
avg_sentiment	        emotional tone
archetype	            human-readable identity
behavior_profile	    machine-readable cognition

In [61]:
# Inspect multiple users from each archetype

for archetype in persona_df["archetype"].unique():

    print("\n" + "="*80)
    print(f"ARCHETYPE: {archetype}")
    print("="*80)

    # Sample 3 users
    sampled_users = (
        persona_df[
            persona_df["archetype"] == archetype
        ]
        .sample(8)
    )

    for _, user_row in sampled_users.iterrows():

        sample_user_id = user_row["user_id"]

        print("\n" + "#"*60)
        print(f"USER ID: {sample_user_id}")
        print("#"*60)

        user_reviews = curated_reviews[
            curated_reviews["user_id"] == sample_user_id
        ]

        # Show 2 reviews
        for i, (_, row) in enumerate(
            user_reviews.head(2).iterrows(),
            start=1
        ):

            print("\n" + "-"*50)
            print(f"Review #{i}")
            print(f"Stars: {row['stars']}")
            print("-"*50)

            print(row["text"][:500])

            print("\n")


ARCHETYPE: Harsh Critic

############################################################
USER ID: eqLx_FhG82BJ08bNAQBnMQ
############################################################

--------------------------------------------------
Review #1
Stars: 5
--------------------------------------------------
Oh my goodness! What have I been missing??? We went to a haunted house the other night and I used Yelp to find a restaurant nearby. Found Cafe Bosna!!! I knew where it was from living near there 10 years ago, so I was not expecting much. Used to be a place called KO's in that strip mall. 

The reviews were overwhelming so I could not resist going there. We had some small sausages for an app. They were served hot off the grill were good. Only so much you can do with just meat though. 

The Chicke



--------------------------------------------------
Review #2
Stars: 1
--------------------------------------------------
Here at 8:15 on Saturday, supposedly opens at 7:30 per the sign. Dark ins

In [ ]:
# STEP 32 — Final Persona Summary
persona_df[
    ["user_id", "archetype"]
].head(10)

,user_id,archetype
0,-EX1hrPRBqNkVavtMllTCA,Harsh Critic
1,-M7fUg7FrdGctKr5f_eMUQ,Emotional Storyteller
2,-WM58wLjtlHlR91xVfM1FQ,Emotional Storyteller
3,-qTtg1D3RidRa4cTB-ftwg,Warm Optimist
4,02H49g16MdRoZKoX6IEoFA,Reactive Reviewer
5,0650daOKAuufqymyOBe3cA,Warm Optimist
6,06Yz-YYYa1U9PN37b6UniA,Emotional Storyteller
7,0Tsu6-uhw_w9Z3W0Xlnazg,Harsh Critic
8,0xJwTzZuWOac7ufeTioeig,Emotional Storyteller
9,175DuOm7IPiEy5f1KG9esA,Harsh Critic


In [ ]:
# STEP 33- Merge Archetypes Back Into Reviews
curated_reviews = curated_reviews.merge(
    persona_df[
        ["user_id", "archetype"]
    ],
    on="user_id",
    how="left"
)

curated_reviews.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date,sentiment,archetype
0,rGI8UdsEGiGeETFiu1VK1w,WJnyWEe_YK7JO47fcovBVw,hRHhP3fhMy3LktPyQa3s_A,3,0,0,0,Good choice in Union Station St. Louis. I had...,2008-08-20 08:07:02,0.204545,Emotional Storyteller
1,Lun9ta1qn_pmuD1VqxiYmg,9gSuVhKyOx3Qn4oI6EQMaA,NwJoFxmYRDxVGXgPtrjQ3w,4,0,0,0,Went there for lunch and was pleasantly surpri...,2009-10-14 21:16:08,0.106667,Emotional Storyteller
2,Wu99UIXo1jGJeu97KCJzsw,iBQKwkuDvAdTM5gLWHgZwg,8uF-bhJFgT4Tn6DTb27viA,4,0,0,0,I don't think there is anything in this place ...,2017-12-16 01:54:03,0.403333,Warm Optimist
3,bY-J5JBKI9m8fiFm4CwCFA,ZybKys6Kg37xX2LMfvcntg,x4XDkWR9fgP4TItqMr8A8A,5,0,0,0,"Delicious, fresh, and friendly volunteers. Yes...",2014-06-11 16:43:45,0.430556,Reactive Reviewer
4,xHwfbcnzpIKXbFvHG8kRTQ,3kvIOBG06_rikfpk-EHIlQ,bXjnfT69E8DJinX-ifOofA,1,31,5,5,I've never had to write a review based on horr...,2012-11-07 17:45:30,0.131566,Emotional Storyteller
